In [1]:
import numpy as np
import pandas as pd
from itertools import combinations

In [2]:
def build_strategy_set(players):
    player_strategies = []
    player_index = []
    for pl in players:
        player_strategies.append(pl.strategy)
        player_index.append(pl.id)

    grids = np.meshgrid(*player_strategies)
    cartesian = np.column_stack([np.ravel(grid) for grid in grids])
    return pd.DataFrame(cartesian, columns=player_index)

def utility(s_i, s_minus_i, player_id, utilities):
    index_labels = s_minus_i.index.tolist()
    query = ""
    for pl_id in index_labels:
        query += f"`{pl_id}` == '{s_minus_i[pl_id]}' &"
    query += f" `{player_id}` == '{s_i}'"
    
    row = utilities.query(query)
    return row[f"payoff_{player_id}"].iloc[0]

def input_strategy_belief(input_text, available_belief):
    while True:
        probability = float(input(f"Probability for {input_text}: "))
        if probability <= available_belief:
            break
        else:
            print(f"Invalid Belief, Available Belief: {available_belief}. Try Again!!")
    return probability

def input_payoff(input_text, num_players):
    while True:
        payoff = input(input_text).split(" ")
        payoff = np.array(payoff, dtype=float)
        if len(payoff) == num_players:
            break
        else:
            print(f"Enter payoffs for {num_players} players.")
    return payoff

In [3]:
class Player:
    def __init__(self, id, strat):
        self.id = id
        self.strategy = np.array(strat, dtype=str)
        self.strategy_set_excluded = pd.DataFrame({})
        self.belief = pd.DataFrame({})
    
    def build_correlated_beliefs(self):
        beliefs = []
        available_belief = 1
        for _, strategy_profile in self.strategy_set_excluded.iterrows():
            probability = input_strategy_belief(strategy_profile, available_belief)            
            available_belief -= probability
            beliefs.append(probability)

        if sum(beliefs) != 1:
            print(f"Invalid Correlated Beliefs Built. Sum: {sum(beliefs)}. Try Again!!")
            self.build_correlated_beliefs(self)

        self.belief = pd.DataFrame(beliefs)
        self.strategy_set_excluded = pd.concat([self.strategy_set_excluded, self.belief], axis=1)

    def build_independent_beliefs(self):
        beliefs = []
        strat_probability = []
        for strat in self.strategy:
            strat_probability.append(input_strategy_belief(strat, 1))
        
        strat_beliefs = pd.DataFrame({
            "strategy": self.strategy,
            "probability": strat_probability
        })
        
        for _, strategy_profile in self.strategy_set_excluded.iterrows():
            belief = 1
            for strat in strategy_profile:
                belief = belief * strat_beliefs[strat]
            beliefs.append(belief)
        
        self.belief = pd.DataFrame(beliefs)
        self.strategy_set_excluded = pd.concat([self.strategy_set_excluded, self.belief], axis=1)

    def find_dominated_strategies(self, utilities):
        for s_i, s_j in list(combinations(self.strategy, 2)):
            i_minus_j = []
            for row in self.strategy_set_excluded.iterrows():
                s_minus_i = row[1] # row is a tuple, row[1] is a pd.Series
                u_s_i = utility(s_i, s_minus_i, self.id, utilities)
                u_s_j = utility(s_j, s_minus_i, self.id, utilities)
                
                i_minus_j.append(u_s_i - u_s_j)
            
            if np.all(np.array(i_minus_j) > 0):
                print(f"Strategy: {s_i} dominates {s_j}")
            elif np.all(np.array(i_minus_j) < 0):
                print(f"Strategy: {s_j} dominates {s_i}")

In [4]:
class Game:
    def __init__(self):
        print("""Game Setup:
              Enter Number of Players(n):
                Finite Positive Integer
              Strategies for each player (Array of strings without spaces):
                ['up', 'down', 'left', 'right']
              Provide Payoffs for all strategy_profiles(s1, s2, ... sn):
                n space separated real numbers""")
        self.players = []
        self.strategy_sets = pd.DataFrame({})
        self.utilities = pd.DataFrame({})

    def setup(self):
        n = int(input("Enter number of players: "))

        for i in range(n):
            player_strategies = input(f"Enter Strategies for Player {i}: ").lower().split(" ")
            self.players.append(Player(id = i, strat = player_strategies))
        
        self.strategy_sets = build_strategy_set(self.players) # Build strategy set for All Players

        print("Enter Payoff for each strategy Profile:")
        payoffs = []
        for _, strategy_profile in self.strategy_sets.iterrows():
            payoff = input_payoff(f"{strategy_profile}: ", n)
            payoffs.append(payoff)
        
        self.utilities = pd.DataFrame(payoffs, columns=[f"payoff_{col}" for col in self.strategy_sets.columns])
        self.utilities = pd.concat([self.strategy_sets, self.utilities], axis = 1)

        for index, pl in enumerate(self.players):
            pl.strategy_set_excluded = build_strategy_set(self.players[:index] + self.players[index+1:])
            # pl.build_correlated_beliefs() or pl.build_independent_beliefs()
    
    def find_dominated_strategies(self):
        for pl in self.players:
            print(f"\nFor Player: {pl.id}")
            pl.find_dominated_strategies(self.utilities)

    def view(self):
        print(f"{len(self.players)} Players")
        print(self.utilities)

In [5]:
game = Game()
game.setup()
game.view()
game.find_dominated_strategies()

Game Setup:
              Enter Number of Players(n):
                Finite Positive Integer
              Strategies for each player (Array of strings without spaces):
                ['up', 'down', 'left', 'right']
              Provide Payoffs for all strategy_profiles(s1, s2, ... sn):
                n space separated real numbers


Enter number of players:  2
Enter Strategies for Player 0:  T M B
Enter Strategies for Player 1:  L C R


Enter Payoff for each strategy Profile:


0    t
1    l
Name: 0, dtype: str:  4 2
0    m
1    l
Name: 1, dtype: str:  1 3
0    b
1    l
Name: 2, dtype: str:  3 3
0    t
1    c
Name: 3, dtype: str:  6 1
0    m
1    c
Name: 4, dtype: str:  5 5
0    b
1    c
Name: 5, dtype: str:  5 2
0    t
1    r
Name: 6, dtype: str:  10 10
0    m
1    r
Name: 7, dtype: str:  9 2
0    b
1    r
Name: 8, dtype: str:  8 8


2 Players
   0  1  payoff_0  payoff_1
0  t  l       4.0       2.0
1  m  l       1.0       3.0
2  b  l       3.0       3.0
3  t  c       6.0       1.0
4  m  c       5.0       5.0
5  b  c       5.0       2.0
6  t  r      10.0      10.0
7  m  r       9.0       2.0
8  b  r       8.0       8.0

For Player: 0
Strategy: t dominates m
Strategy: t dominates b

For Player: 1
